In [0]:
storage_account_name = "creditstorage2026"
# Secret key for storage account
storage_account_key = dbutils.secrets.get(scope="credit-risk-scope", key="storage-key")
container_name = "raw"

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

In [0]:
# Verify connection by listing raw container
dbutils.fs.ls(f"abfss://raw@{storage_account_name}.dfs.core.windows.net/")

In [0]:
raw_path = f"abfss://raw@{storage_account_name}.dfs.core.windows.net/credit_risk_dataset.csv"

# Load data from csv
df_raw = spark.read.csv(raw_path, header=True, inferSchema=True)

df_raw.printSchema()

In [0]:
# Null check

from pyspark.sql.functions import col, count, when, isnan

# Check nulls separately for numeric and string columns
numeric_cols = [f.name for f in df_raw.schema.fields 
                if str(f.dataType) in ('IntegerType()', 'DoubleType()')]
string_cols = [f.name for f in df_raw.schema.fields 
               if str(f.dataType) == 'StringType()']

# Null check for numeric columns (can use isnan)
print("Numeric column nulls:")
df_raw.select([
    count(when(col(c).isNull() | isnan(c), c)).alias(c)
    for c in numeric_cols
]).show()

# Null check for string columns (isNull only)
print("String column nulls:")
df_raw.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in string_cols
]).show()

In [0]:
# Transformation Layer

from pyspark.sql.functions import col, when, mean

# Calculate means for imputation
mean_emp_length = df_raw.select(mean(col("person_emp_length"))).collect()[0][0]
mean_int_rate = df_raw.select(mean(col("loan_int_rate"))).collect()[0][0]

df_transformed = df_raw \
    .fillna({"person_emp_length": mean_emp_length,
             "loan_int_rate": mean_int_rate}) \
    .withColumn("loan_status_label",
             when(col("loan_status") == 1, "Default")
            .otherwise("Non-Default")) \
    .withColumn("high_risk",
             when((col("loan_int_rate") > 15) & 
                  (col("loan_percent_income") > 0.3), True)
            .otherwise(False))

# Verify no nulls remain
df_transformed.select("person_emp_length", "loan_int_rate").describe().show()

In [0]:
# Outlier filter (max employed length was 123 which is not possible)

df_transformed = df_transformed \
    .filter(col("person_emp_length") <= 60) \
    .filter(col("person_age") <= 100)

print(f"Rows after outlier removal: {df_transformed.count()}")

In [0]:
processed_path = f"abfss://processed@{storage_account_name}.dfs.core.windows.net/credit_risk_delta"

df_transformed.write \
    .format("delta") \
    .mode("overwrite") \
    .save(processed_path)

print("Delta table written successfully")

In [0]:
df_verify = spark.read.format("delta").load(processed_path)

print(f"Row count: {df_verify.count()}")
print(f"Columns: {df_verify.columns}")
df_verify.show(5)